In [1]:
import numpy as np
import matplotlib
import json
import torch
from PIL import Image
from pillow_heif import register_heif_opener
import requests

register_heif_opener()

def overlay_masks(image, masks):
    image = image.convert("RGBA")
    masks = 255 * masks.cpu().numpy().astype(np.uint8)
    
    n_masks = masks.shape[0]
    cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
    colors = [
        tuple(int(c * 255) for c in cmap(i)[:3])
        for i in range(n_masks)
    ]

    for mask, color in zip(masks, colors):
        mask = Image.fromarray(mask)
        overlay = Image.new("RGBA", image.size, color + (0,))
        alpha = mask.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        image = Image.alpha_composite(image, overlay)
    return image


In [ ]:
import math
from pathlib import Path
import matplotlib.pyplot as plt

results_dir = Path("results")
metadata_files = sorted(results_dir.glob("*.json"))

if not metadata_files:
    raise FileNotFoundError(f"No result metadata JSON files found in {results_dir.resolve()}")

visualizations = []

for metadata_path in metadata_files:
    with metadata_path.open() as f:
        metadata = json.load(f)["metadata"]

    npz_path = results_dir / metadata["npz_filename"]
    mask_data = np.load(npz_path)
    masks = mask_data["masks"]

    if masks.ndim == 2:
        masks = masks[None, ...]

    image_path = Path(metadata["image_path"])
    candidate_image_paths = [
        image_path,
        Path.cwd() / image_path.relative_to("/") if image_path.is_absolute() else Path.cwd() / image_path,
        Path("images") / metadata["image_filename"],
        metadata_path.parent / metadata["image_filename"],
    ]

    resolved_image_path = next((path for path in candidate_image_paths if path.exists()), None)
    if resolved_image_path is None:
        searched_paths = "\n".join(str(path) for path in candidate_image_paths)
        raise FileNotFoundError(
            f"Could not find image for {metadata['image_filename']}. Tried:\n{searched_paths}"
        )

    image = Image.open(resolved_image_path).convert("RGB")

    if masks.size == 0 or masks.shape[0] == 0:
        overlay_image = image
    else:
        overlay_image = overlay_masks(image, torch.from_numpy(masks))

    visualizations.append((overlay_image, metadata))

n_cols = 3
n_rows = math.ceil(len(visualizations) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, (overlay_image, metadata) in zip(axes, visualizations):
    ax.imshow(overlay_image)
    ax.set_title(f"{metadata['image_filename']}\nprompt: {metadata['text_prompt']}")
    ax.axis("off")

for ax in axes[len(visualizations):]:
    ax.axis("off")

plt.tight_layout()
plt.show()
